# Module 3 - Class 5 Assignment: Pipelines
**Khamidullokhon Abduvokhidov**

In [ ]:
# Load Telco data and create a binary churn target.
import pandas as pd
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [ ]:
# Choose features and create reproducible training and test sets.
num_features = ['tenure', 'MonthlyCharges', 'SeniorCitizen']
cat_features = ['Contract', 'InternetService', 'PaymentMethod']
X = df[num_features + cat_features]
y = df['Churn_bin']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Impute numeric values and scale them into the zero-to-one range.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
]) 

In [ ]:
# Impute categorical values and turn categories into indicator columns.
from sklearn.preprocessing import OneHotEncoder
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
]) 

In [ ]:
# Route each feature type through its appropriate preprocessing pipeline.
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
]) 

In [ ]:
# Assemble preprocessing and a random forest in one pipeline.
from sklearn.ensemble import RandomForestClassifier
full_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])
full_pipe

In [ ]:
# Fit every preprocessing step and the model together.
full_pipe.fit(X_train, y_train)
print('Trained.')

In [ ]:
# Score the trained pipeline against held-out customer data.
acc = full_pipe.score(X_test, y_test)
print(f'Random Forest test accuracy: {acc:.4f}')

In [ ]:
# Save the full pipeline and confirm that the reloaded model works.
import joblib
joblib.dump(full_pipe, 'rf_pipeline.joblib')
loaded = joblib.load('rf_pipeline.joblib')
print('Loaded model test accuracy:', round(loaded.score(X_test, y_test), 4))